# **NEUROCHIP SMART FAULT MAPPER**

### Detect and pinpoint faults in a VLSI design using a Graphical Neural Network.

Author: Nikhil Bhaktha

GitHub Profile: [GlobosNik](https://github.com/GlobosNik)

---
## **Data Extractor Notebook**

## Load the required python modules

Run the following commands in the bash terminal.

```bash
# Install Pyverilog for parsing
pip install pyverilog

# Install iverilog for preprocessing Verilog files (dependency for Pyverilog)
sudo apt-get install -y iverilog
```

Import the required python dependencies.

In [ ]:
# Modules for VLSI analysis
import pandas as pd
import re
from tqdm import tqdm
import os
from collections import defaultdict, Counter
from pathlib import Path
from pyverilog.vparser.parser import parse
import concurrent.futures
import threading
import functools


## Clone the HDL Benchmarks Repository

Clone the HDL Benchmarks GitHub repository to access the dataset.

GitHub Link: [HDL_Benchmarks](https://github.com/ispras/hdl-benchmarks)

```bash
# Remove directory if it exists from previous runs
rm -rf HDL_Benchmarks

# Clone the Benchmarks repository
git clone https://github.com/ispras/hdl-benchmarks {'HDL_Benchmarks'}
```


## Normalize and Encode each `.v` File

This function extracts the signal names from declarations/gates/assigns etc. and encodes them into a standard format.
* It takes a raw Verilog text, removes comments, identifies all signal names from declarations (like `wire`, `reg`, `input`, `output`), gates (like `and`, `or`), and assignments (`assign`).
* It then creates a mapping to replace these original signal names with a standardized format, `S0`, `S1`, `S2` etc.
* This standardization is crucial for consistent processing and analysis of VLSI designs.
* The regular expressions at the beginning are pre-compiled patterns used by the function to efficiently find and extract these different types of signal declarations and usages.

In [ ]:
# Pre-compile regex patterns for verilogEncoder
_re_text_no_comments = re.compile(r'/\*.*?\*/|//.*?(?=\n|$)', flags=re.DOTALL | re.MULTILINE)
_re_decl_pattern = re.compile(r'(?:wire|reg|input|output)\s*(?:\\[.*?\\])?\s*([a-zA-Z_]\w*(?:\s*,\s*[a-zA-Z_]\w*)*?)(?=\s*;)')
_re_gate_pattern = re.compile(r'(?:and|or|not|xor|nor|xnor|buf|inv)\s*\(\s*([^)]+)\)')
_re_assign_pattern = re.compile(r'assign\s+([a-zA-Z_]\w*)\s*=\s*([^;]+);')
_re_findall_signals = re.compile(r'[a-zA-Z_]\w+')

def verilogEncoder(text):
    text_no_comments = _re_text_no_comments.sub('', text)

    # Extract signals (decls + gates/assigns)
    decl_pattern = _re_decl_pattern
    gate_pattern = _re_gate_pattern
    assign_pattern = _re_assign_pattern

    signals = set()
    for pat_compiled in [_re_decl_pattern, _re_gate_pattern, _re_assign_pattern]:
        for match in pat_compiled.finditer(text_no_comments):
            # Determine which pattern matched to extract correct groups
            if pat_compiled == _re_assign_pattern:
                lhs = match.group(1)
                rhs = match.group(2)
                signals.add(lhs)
                sigs_rhs = _re_findall_signals.findall(rhs)
                signals.update(sigs_rhs)
            else:
                args = match.group(1) if len(match.groups()) > 0 else ''
                sigs = _re_findall_signals.findall(args)
                signals.update(sigs)

    sorted_signals = sorted(list(signals))
    signal_map = {sig: f'S{i}' for i, sig in enumerate(sorted_signals)}

    normalized = text_no_comments
    for orig, enc in signal_map.items():
        normalized = re.compile(rf'\b{re.escape(orig)}\b').sub(enc, normalized)

    return normalized, signal_map

## Parse the Files and Convert to DataFrames

The `verilogExtractor` function is defined here, which takes normalized Verilog code and converts it into GNN representation suitable for analysis.

- **Regular Expressions:**
    - The pre-compiled regular expressions (`_re_gate_extract_pat`, `_re_assign_extract_pat`, `_re_rhs_signals`, `_re_port_extract_pat`) are defined here. 
    - These patterns are used to efficiently identify and extract information about gates (like `and`, `or`), signal assignments (`assign`), signals on the right-hand side of assignments, and input/output port declarations from the normalized Verilog code.
- **Extractor Function:** 
    It initializes empty lists for nodes and edges (which will eventually become dataframes), and a dictionary `graph_stats` to store various metrics about the circuit.
- **Signal ID Mapping:** 
    - A `signal_to_id` dictionary and `next_id` counter are used to assign a unique numerical ID to each signal (`S0`, `S1`, etc.) found in the Verilog code. 
    - The `get_id` helper function ensures that each signal gets a unique ID and consistently maps the same signal name to the same ID.
- **Parsing Gates:** 
    - The code iterates through all matches of `_re_gate_extract_pat` in the normalized Verilog. 
    - For each gate, it extracts the gate type (`gtype`), output signal (`out_sig`), and input signals (`ins_str`). 
    - It then creates a node for the output signal with its type, `fan-in` (number of inputs), and initializes `fan-out` to 0. 
    - Edges are created from each input signal to the output signal. 
    - The `graph_stats` are updated with the count of gates and gate types.
- **Parsing Assignments:** 
    - Similar to gates, this parses `assign` statements using `_re_assign_extract_pat`. 
    - It extracts the output signal and the right-hand side (RHS) expression. 
    - It then uses `_re_rhs_signals` to find all signals on the RHS, treating them as inputs to the assignment.
- **Parsing Ports:** 
    - The `_re_port_extract_pat` is used to identify input and output port declarations. 
    - For each port, a node is created with the appropriate type (`input` or `output`) and `is_port flag` set to 1. The `num_inputs` and `num_outputs` in `graph_stats` are updated.
- **Compute Fanout, Degrees, and Depth:** After all nodes and edges are collected, the function calculates the fan-out for each node by counting how many times a node appears as a source in the edges list.

After this, the final dataframes for nodes and edges are returned.

In [ ]:
# Pre-compile regex patterns for verilogExtractor
_re_gate_extract_pat = re.compile(r'(and|or|not|xor|nor|xnor|buf|inv)\s*\(\s*([A-Z]\d+)\s*,?\s*([^)]*)\)\s*;')
_re_assign_extract_pat = re.compile(r'assign\s+([A-Z]\d+)\s*=\s*([^;]+);')
_re_rhs_signals = re.compile(r'[A-Z]\d+')
_re_port_extract_pat = re.compile(r'(input|output)\s+([A-Z]\d+(?:\s*,\s*[A-Z]\d+)*);')

# Parse normalized Verilog into GCN features
def verilogExtractor(normalized_code):
    nodes = []  # [node_id, type, fanin, fanout, is_port]
    edges = []  # [src_id, dst_id]
    graph_stats = {
        'num_gates': 0, 'num_inputs': 0, 'num_outputs': 0,
        'gate_types': Counter(), 'avg_degree': 0, 'max_depth': 0
    }

    signal_to_id = {}  # S0 -> 0
    next_id = 0

    def get_id(sig):
        nonlocal next_id
        if sig not in signal_to_id:
            signal_to_id[sig] = next_id
            next_id += 1
        return signal_to_id[sig]

    # Parse gates
    for match in _re_gate_extract_pat.finditer(normalized_code):
        gtype, out_sig, ins_str = match.groups()
        out_id = get_id(out_sig)
        ins = [s.strip() for s in ins_str.split(',') if s.strip()]
        in_ids = [get_id(sig) for sig in ins]

        nodes.append({'node_id': out_id, 'type': gtype, 'fanin': len(ins), 'fanout': 0, 'is_port': 0})
        graph_stats['gate_types'][gtype] += 1
        graph_stats['num_gates'] += 1

        for in_id in in_ids:
            edges.append({'src': in_id, 'dst': out_id})

    # Parse assigns: assign S1 = S2 & S3;
    for match in _re_assign_extract_pat.finditer(normalized_code):
        out_sig, rhs = match.groups()
        out_id = get_id(out_sig)
        # Simple: extract signals in RHS as inputs
        ins = _re_rhs_signals.findall(rhs)
        in_ids = [get_id(sig) for sig in ins]

        nodes.append({'node_id': out_id, 'type': 'assign', 'fanin': len(ins), 'fanout': 0, 'is_port': 0})
        graph_stats['num_gates'] += 1

        for in_id in in_ids:
            edges.append({'src': in_id, 'dst': out_id})

    # Ports (inputs/outputs)
    for match in _re_port_extract_pat.finditer(normalized_code):
        ptype, ports_str = match.groups()
        ports = [s.strip() for s in ports_str.split(',') if s.strip()]
        for port in ports:
            pid = get_id(port)
            if ptype == 'input':
                graph_stats['num_inputs'] += 1
                nodes.append({'node_id': pid, 'type': 'input', 'fanin': 0, 'fanout': 0, 'is_port': 1})
            else:
                graph_stats['num_outputs'] += 1
                nodes.append({'node_id': pid, 'type': 'output', 'fanin': 0, 'fanout': 0, 'is_port': 1})

    # Compute fanout, degrees, depth
    fanout_count = defaultdict(int)
    for e in edges:
        fanout_count[e['src']] += 1

    for node in nodes:
        node['fanout'] = fanout_count[node['node_id']]

    if nodes:
        degrees = [n['fanin'] + n['fanout'] for n in nodes]
        graph_stats['avg_degree'] = sum(degrees) / len(nodes)
        graph_stats['max_depth'] = max(degrees)

    return pd.DataFrame(nodes), pd.DataFrame(edges), graph_stats

## Process the Verilog Files

* The `verilogProcessor` function takes a filepath to a Verilog file, along with a `circuit_id_counter` and `circuit_id_lock` for thread-safe unique ID generation.
* Its main purpose is to read a Verilog file, normalize its signal names, extract graph features (nodes and edges), and then compile this information into dataframes.
* It handles potential encoding errors when reading files and uses helper functions like `verilogEncoder` and `verilogExtractor` for the core processing.
* It assigns a unique `circuit_id` to each processed file and includes error handling for any issues that arise during processing.

The verilog files are extracted from the cloned GitHub repository and are appended into a list for the extractor functions to iterate upon.

In [ ]:
def verilogProcessor(filepath, circuit_id_counter, circuit_id_lock):
    status_row = {}
    nodes_df = pd.DataFrame()
    edges_df = pd.DataFrame()

    # Get a unique circuit_id
    with circuit_id_lock:
        circuit_id = circuit_id_counter.value
        circuit_id_counter.value += 1

    status_row['circuit_id'] = circuit_id
    status_row['file'] = os.path.basename(filepath)
    status_row['path'] = filepath

    try:
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                verilog_code = f.read()
        except UnicodeDecodeError:
            with open(filepath, 'r', encoding='latin-1') as f:
                verilog_code = f.read()

        # Encode the Verilog file
        normalized_code, signal_map = verilogEncoder(verilog_code)
        status_row['signal_map_len'] = len(signal_map)

        # Extract graph data
        nodes, edges, graph_stats = verilogExtractor(normalized_code)

        if not nodes.empty:
            nodes['circuit_id'] = circuit_id
            nodes_df = nodes
        if not edges.empty:
            edges['circuit_id'] = circuit_id
            edges_df = edges

        status_row.update(graph_stats)

    except Exception as e:
        status_row['error'] = str(e)
        print(f"Error processing {filepath}: {e}")

    return status_row, nodes_df, edges_df

print("Defined `verilogProcessor` function.")


# Find all .v files in the HDL_Benchmarks directory
verilog_fileList = list(Path('HDL_Benchmarks').rglob('*.v'))
print(f"Found {len(verilog_fileList)} Verilog files to process.")

## Node Feature Engineering

Implement one-hot encoding for the dataframe parameters, using a `oneHotEncoder` function, which is responsible for transforming and normalizing features in the `nodesDF` dataframe, passed as its parameter.

It performs feature engineering by applying one-hot encoding to the `type` column, creating new binary columns for each gate type. It also normalizes numerical features like `fanin` and `fanout` by scaling them between 0 and 1, and then calculates a normalized degree as the average of the normalized fan-in and fan-out.

In [ ]:
# One-hot encoding for gate types
gate_types = ['input', 'output', 'assign', 'and', 'or', 'not', 'xor', 'nor', 'xnor', 'buf', 'inv']
type_to_idx = {t: i for i, t in enumerate(gate_types)}

def oneHotEncoder(nodesDF):

    onehotEncodedCols = []
    for i in tqdm(range(len(gate_types))):
        col_name = f'type_{gate_types[i]}'
        onehotEncodedCols.append(col_name)
        nodesDF[col_name] = (nodesDF['type'] == gate_types[i]).astype(int)

    nodesDF[onehotEncodedCols] = nodesDF[onehotEncodedCols].fillna(0).astype(int)

    # Normalize numerics
    nodesDF['fanin_norm'] = nodesDF['fanin'] / (nodesDF['fanin'].max() + 1e-6)
    nodesDF['fanout_norm'] = nodesDF['fanout'] / (nodesDF['fanout'].max() + 1e-6)
    nodesDF['degree_norm'] = (nodesDF['fanin_norm'] + nodesDF['fanout_norm']) / 2

    return nodesDF

## Multithreading Setup for Faster Data Extraction

This function defines the multithreading process for extracting and transforming data from the verilog (`.v`) files.

*   The `ThreadSafeInt` class manages a shared integer counter across multiple threads preventing race conditions.
*   The `verilogProcessor` function is prepared for multithreading using `functools.partial`, binding the thread-safe counter and lock.
*   It calculates an optimal number of worker threads and defines the `batchExtractor` function. This function takes a list of Verilog file paths and a batch number.
*   It uses `ThreadPoolExecutor` to process files concurrently, collecting status, node and edge data and storing it as a dataframe after applying one-hot encoding, following which they are stored as `.csv` files.


In [ ]:
# Define a thread-safe integer class
class ThreadSafeInt:
    def __init__(self, initial_value=0):
        self._value = initial_value
        self._lock = threading.Lock()

    @property
    def value(self):
        with self._lock:
            return self._value

    @value.setter
    def value(self, new_value):
        with self._lock:
            self._value = new_value

circuit_id_counter = ThreadSafeInt(0)
circuit_id_lock = threading.Lock()

# Prepare arguments for multithreading
# functools.partial will bind circuit_id_counter and circuit_id_lock to verilogProcessor
partialProcess_func = functools.partial(verilogProcessor, circuit_id_counter=circuit_id_counter, circuit_id_lock=circuit_id_lock)

# Determine the number of workers. It is capped at 32 or number of files.
num_workers = min(os.cpu_count() * 2, 32, len(verilog_fileList)) if len(verilog_fileList) > 0 else 1
if num_workers == 0: num_workers = 1    # Ensure at least one worker is enabled

# Define the extraction process
def batchExtractor(verilog_files, batch):

    # Initialize empty lists to store results from each process
    all_status_rows = []
    all_nodes_dfs = []
    all_edges_dfs = []

    # Use concurrent.futures.ThreadPoolExecutor for multithreading
    if verilog_files:
        with concurrent.futures.ThreadPoolExecutor(max_workers=num_workers) as executor:
            # Submit tasks and collect futures for the batch
            futures = [executor.submit(partialProcess_func, filepath) for filepath in verilog_files]

            # Process results as they complete
            for future in tqdm(concurrent.futures.as_completed(futures), total=len(verilog_files), desc=f"Processing Verilog files - Batch {batch}"):
                try:
                    result_status, result_nodes_df, result_edges_df = future.result()
                    all_status_rows.append(result_status)
                    if not result_nodes_df.empty:
                        all_nodes_dfs.append(result_nodes_df)
                    if not result_edges_df.empty:
                        all_edges_dfs.append(result_edges_df)
                except Exception as exc:
                    # Log any exceptions that occurred in the thread
                    print(f"A file processing generated an exception: {exc}")
    else:
        print("No .v files found in HDL_Benchmarks to process.")

    # Concatenate results into final DataFrames
    if all_status_rows:
        statusDF = pd.DataFrame(all_status_rows)
        # Handle cases where Counter objects might have been stored as a string
        statusDF['gate_types'] = statusDF['gate_types'].apply(lambda x: eval(x) if isinstance(x, str) else x)
    else:
        statusDF = pd.DataFrame()

    if all_nodes_dfs:
        nodesDF = pd.concat(all_nodes_dfs, ignore_index=True)
    else:
        nodesDF = pd.DataFrame()

    if all_edges_dfs:
        edgesDF = pd.concat(all_edges_dfs, ignore_index=True)
    else:
        edgesDF = pd.DataFrame()

    print(f"\n--- Processing Complete for batch {batch} ---")

    # Display results (head of each DataFrame)
    print("\nstatusDF Head:\n")
    if not statusDF.empty:
        print(statusDF.head())

    else:
        print("statusDF is empty.")

    print("\nnodesDF Head:\n")
    if not nodesDF.empty:
        print(nodesDF.head())
    else:
        print("nodesDF is empty.")

    print("\nedgesDF Head:\n")
    if not edgesDF.empty:
        print(edgesDF.head())
    else:
        print("edgesDF is empty.")

    # Implement one hot encoding
    nodesDF = oneHotEncoder(nodesDF)

    # Print random 5 elements from dataframe
    print("\n--- Random 5 elements from nodesDF ---")
    print(nodesDF.sample(5))
    print("\n--- Random 5 elements from edgesDF ---")
    print(edgesDF.sample(5))
    print("\n--- Random 5 elements from statusDF ---")
    print(statusDF.sample(5))

    # Save the dataframes
    statusDF.to_csv(f'statusData_{batch}.csv', index=False)
    nodesDF.to_csv(f'nodesData_{batch}.csv', index=False)
    edgesDF.to_csv(f'edgesData_{batch}.csv', index=False)

Batch 1 Extraction

In [ ]:
# Extract first 25% of features in batch 1
iterationCount = int(len(verilog_fileList) * 0.25)

batchExtractor(verilog_fileList[:iterationCount], batch=1)

Batch 2 Extraction

In [ ]:
# Extract next 25% of features in batch 2
batchExtractor(verilog_fileList[iterationCount:iterationCount*2], batch=2)

Batch 3 Extraction

In [ ]:
# Extract next 25% of features in batch 3
batchExtractor(verilog_fileList[iterationCount*2:iterationCount*3], batch=3)

Batch 4 Extraction

In [ ]:
# Extract next 25% of features in batch 4
batchExtractor(verilog_fileList[iterationCount*3:], batch=4)